In [1]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
from form990_parser_old import (
    NAMESPACE,
    RETURN_DATA_PATH
)
import re
import json

In [55]:
example_root = Path('example_xmls')

In [56]:
df = pd.read_csv('example_usecase_lookup.csv')
df.head()

,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
0,A 501c3 filed a 990 containing IRS990 in 2018.,136227614,HADASSAH GROUP RETURN,501c3,xml\2019\2019_03A\201913199349306206_public.xml,2018,990,1.0,3652.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,A 501c3 filed a 990 containing IRS990ScheduleA...,135562401,YOUNG MEN'S CHRISTIAN ASSOCIATION RETIREMENT FUND,501c3,xml\2019\2019_03A\201913159349303926_public.xml,2018,990,1.0,267.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,A 501c3 filed a 990 containing IRS990ScheduleB...,452318579,CARIBBEAN STUDIES ASSOCIATIONINC,501c3,xml\2021\2021_01A\202101069349300000_public.xml,2018,990,1.0,190.0,1.0,...,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
3,A 501c3 filed a 990 containing IRS990ScheduleD...,621577879,OBION COUNTY HABITAT FOR HUMANITY,501c3,xml\2019\2019_01A\201901349349305675_public.xml,2018,990,1.0,189.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,A 501c3 filed a 990 containing IRS990ScheduleF...,510198509,TIDES FOUNDATION,501c3,xml\2019\2019_03A\201913189349314251_public.xml,2018,990,1.0,274.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [57]:
c4s = df.loc[df.OrgType == '501c4']
c4_ScC = c4s.loc[c4s.situation.str.contains('ScheduleC')]
for i, (situation, file) in enumerate(zip(c4_ScC.loc[:, 'situation'].values, c4_ScC.loc[:, 'file'].values)):
    print(i, situation, file)

0 A 501c4 filed a 990 containing IRS990ScheduleC in 2018. xml\2019\2019_02A\201903159349302410_public.xml
1 A 501c4 filed a 990EZ containing IRS990ScheduleC in 2018. xml\2019\2019_03A\201911359349201176_public.xml
2 A 501c4 filed a 990 containing IRS990ScheduleC in 2017. xml\2019\2019_02A\201910719349300951_public.xml
3 A 501c4 filed a 990EZ containing IRS990ScheduleC in 2017. xml\2019\2019_01A\201902059349200805_public.xml
4 A 501c4 filed a 990 containing IRS990ScheduleC in 2016. xml\2019\2019_03A\201912039349300116_public.xml
5 A 501c4 filed a 990EZ containing IRS990ScheduleC in 2016. xml\2019\2019_02A\201911159349200146_public.xml
6 A 501c4 filed a 990 containing IRS990ScheduleC in 2019. xml\2020\2020_03A\202013179349305666_public.xml
7 A 501c4 filed a 990EZ containing IRS990ScheduleC in 2019. xml\2021\2021_01A\202140119349200504_public.xml
8 A 501c4 filed a 990 containing IRS990ScheduleC in 2020. xml\2021\2021_01A\202113199349304936_public.xml
9 A 501c4 filed a 990EZ containing IRS

In [58]:
c4_ScC.loc[:, ["situation", "EIN", "Name", "OrgType", "file", "TaxYr", "ReturnTypeCd", "IRS990ScheduleC", "IRS990ScheduleCSize"]].sort_values(by="IRS990ScheduleCSize", ascending=False)

,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990ScheduleC,IRS990ScheduleCSize
931,A 501c4 filed a 990 containing IRS990ScheduleC...,208802884,EVERYTOWN FOR GUN SAFETY ACTION FUND,501c4,xml\2025\2025_11B\202543189349303624_public.xml,2024,990,1.0,258.0
467,A 501c4 filed a 990 containing IRS990ScheduleC...,208802884,EVERYTOWN FOR GUN SAFETY ACTION FUND,501c4,xml\2021\2021_01A\202113199349304936_public.xml,2020,990,1.0,167.0
34,A 501c4 filed a 990 containing IRS990ScheduleC...,870409820,SELECTHEALTH INC,501c4,xml\2019\2019_02A\201903159349302410_public.xml,2018,990,1.0,75.0
348,A 501c4 filed a 990 containing IRS990ScheduleC...,540844477,DELTA DENTAL OF VIRGINIA,501c4,xml\2020\2020_03A\202013179349305666_public.xml,2019,990,1.0,73.0
546,A 501c4 filed a 990EZ containing IRS990Schedul...,113507566,NASSAU COUNTY CORRECTION OFFICERS PAC,501c4,xml\2021\2021_01A\202143219349200104_public.xml,2020,990EZ,1.0,69.0
1009,A 501c4 filed a 990EZ containing IRS990Schedul...,113507566,NASSAU COUNTY CORRECTION OFFICERS PAC,501c4,xml\2025\2025_11B\202533109349202138_public.xml,2024,990EZ,1.0,54.0
426,A 501c4 filed a 990EZ containing IRS990Schedul...,113507566,NASSAU COUNTY CORRECTION OFFICERS PAC,501c4,xml\2021\2021_01A\202140119349200504_public.xml,2019,990EZ,1.0,51.0
815,A 501c4 filed a 990 containing IRS990ScheduleC...,10879928,CHESAPEAKE CLIMATE ACTION NETWORK ACTION FUND INC,501c4,xml\2025\2025_05A\202521219349301927_public.xml,2023,990,1.0,51.0
587,A 501c4 filed a 990 containing IRS990ScheduleC...,461457252,ARDA RESORT OWNERS COALITION,501c4,xml\2022\2022_01A\202221109349300612_public.xml,2021,990,1.0,39.0
149,A 501c4 filed a 990 containing IRS990ScheduleC...,461457252,ARDA RESORT OWNERS COALITION,501c4,xml\2019\2019_02A\201910719349300951_public.xml,2017,990,1.0,36.0


In [60]:
schec_c_elems = set()
for file in c4_ScC.loc[:, 'file'].values:
    print(file)
    file = re.sub(r'^xml', r'example_xmls', file)
    file = re.sub(r'\\', '/', file)
    schec_c = ET.parse(file).getroot().find(f'.//{NAMESPACE}IRS990ScheduleC')
    
    curr_content = set([elem.tag.replace(NAMESPACE, '') for elem in schec_c.iter()])
    
    diff = curr_content - schec_c_elems
    schec_c_elems |= curr_content
    for new_elem in diff:
        print(new_elem)
    print("="*20)

xml\2019\2019_02A\201903159349302410_public.xml
StateAbbreviationCd
Form1120POLFiledInd
Expended527ActivitiesAmt
TotalExemptFunctionExpendAmt
PoliticalExpendituresAmt
InternalFundsContributedAmt
IRS990ScheduleC
Section527PoliticalOrgGrp
ZIPCd
ExplanationTxt
ContributionsRcvdDlvrAmt
BusinessNameLine1Txt
OrganizationBusinessName
FormAndLineReferenceDesc
CityNm
SupplementalInformationDetail
AddressLine1Txt
USAddress
PaidInternalFundsAmt
xml\2019\2019_03A\201911359349201176_public.xml
Form4720FiledInd
AgreeCarryoverPriorYearInd
RalliesDemonstrationsInd
VolunteersInd
SubstantiallyAllDuesNondedInd
DirectContactLegislatorsInd
MediaAdvertisementsInd
Form4720FiledSection4955TaxInd
OnlyInHouseLobbyingInd
CorrectionMadeInd
OtherActivitiesInd
PaidStaffOrManagementInd
NotDescribedSection501c3Ind
GrantsOtherOrganizationsInd
MailingsMembersInd
PublicationsOrBroadcastInd
Form4720Filed4912TaxInd
xml\2019\2019_02A\201910719349300951_public.xml
EIN
xml\2019\2019_01A\201902059349200805_public.xml
Voluntee

In [37]:
# ! NOTE: Some 501c4's filled out 501c3 sections and vice versa
schedule_c = {
    'Part I': {
        'Title': '',
        'Stipulation': 'Complete if the organization is exempt under section 501(c) or is a section 527 organization.',
        'OrgTypes': [],
        'Flat Content': {
            'Part I-A': {
                'Title': '',
                'Stipulation': '',
                'OrgTypes': [r'501c\d{1}', '527'],
                'Flat Content': {
                    # Key format: (Order, Element Tag)
                    (0, 'PoliticalExpendituresAmt'): 'PoliticalExpendituresAmt',
                    (1, 'VolunteerHoursCnt'): 'VolunteerHoursCnt',
                }
            },
            'Part I-B': {
                'Title': '',
                'Stipulation': 'Complete if the organization is exempt under section 501(c)(3).',
                'OrgTypes': ['501c3'],
                'Flat Content': {
                    # Key format: (Order, Element Tag)
                    (0, 'Section4955OrganizationTaxAmt'): 'Section4955OrganizationTaxAmt',
                    (1, 'Section4955ManagersTaxAmt'): 'Section4955ManagersTaxAmt',
                    (2, 'Form4720FiledSection4955TaxInd'): 'CorrectionMadeInd',
                    (3, 'CorrectionMadeInd'): 'CorrectionMadeInd',
                }
            },
            'Part I-C': {
                'Title': '',
                'Stipulation': 'Complete if the organization is exempt under section 501(c), except section 501(c)(3).',
                'OrgTypes': [r'501c4'],
                'Flat Content': {
                    # Key format: (Order, Element Tag)
                    (0, 'Expended527ActivitiesAmt'): 'Expended527ActivitiesAmt',
                    (1, 'InternalFundsContributedAmt'): 'InternalFundsContributedAmt',
                    (2, 'TotalExemptFunctionExpendAmt'): 'TotalExemptFunctionExpendAmt',
                    (3, 'Form1120POLFiledInd'): 'Form1120POLFiledInd',
                },
                'Nested Content': [
                    'Section527PoliticalOrgGrp'
                ]
            },
        }
    },
    'Part II': {
        'Title': '',
        'Stipulation': '',
        'OrgTypes': [],
        'Flat Content': {
            'Part II-A': {
                'Title': '',
                'Stipulation': 'Complete if the organization is exempt under section 501(c)(3) and filed Form 5768 (election under section 501(h)).',
                'OrgTypes': [r'501c\d{1}', '527'],
                'Flat Content': {
                    # Key format: (Order, Element Tag)
                    (0, 'TotalGrassrootsLobbyingGrp'): {
                        # Key format: (Order, Element Tag)
                        (0, 'FilingOrganizationsTotalAmt'): 'FilingOrganizationsTotalGrassrootsLobbyingAmt',
                        (1, 'AffiliatedGroupTotalAmt'): 'AffiliatedGroupTotalGrassrootsLobbyingAmt',
                    },
                    (1, 'TotalDirectLobbyingGrp'): {
                        # Key format: (Order, Element Tag)
                        (0, 'FilingOrganizationsTotalAmt'): 'FilingOrganizationsTotalDirectLobbyingAmt',
                        (1, 'AffiliatedGroupTotalAmt'): 'AffiliatedGroupTotalDirectLobbyingAmt',
                    },
                    (2, 'TotalLobbyingExpendGrp'): {
                        # Key format: (Order, Element Tag)
                        (0, 'FilingOrganizationsTotalAmt'): 'FilingOrganizationsTotalLobbyingExpendAmt',
                        (1, 'AffiliatedGroupTotalAmt'): 'AffiliatedGroupTotalLobbyingExpendAmt',
                    },
                    (3, 'OtherExemptPurposeExpendGrp'): {
                        # Key format: (Order, Element Tag)
                        (0, 'FilingOrganizationsTotalAmt'): 'FilingOrganizationsTotalOtherExemptPurposeExpendAmt',
                        (1, 'AffiliatedGroupTotalAmt'): 'AffiliatedGroupTotalOtherExemptPurposeExpendAmt',
                    },
                    (4, 'TotalExemptPurposeExpendGrp'): {
                        # Key format: (Order, Element Tag)
                        (0, 'FilingOrganizationsTotalAmt'): 'FilingOrganizationsTotalExemptPurposeExpendAmt',
                        (1, 'AffiliatedGroupTotalAmt'): 'AffiliatedGroupTotalExemptPurposeExpendAmt',
                    },
                    (5, 'LobbyingNontaxableAmountGrp'): {
                        # Key format: (Order, Element Tag)
                        (0, 'FilingOrganizationsTotalAmt'): 'FilingOrganizationsTotalLobbyingNontaxableAmt',
                        (1, 'AffiliatedGroupTotalAmt'): 'AffiliatedGroupTotalLobbyingNontaxableAmt',
                    },
                    (6, 'GrassrootsNontaxableGrp'): {
                        # Key format: (Order, Element Tag)
                        (0, 'FilingOrganizationsTotalAmt'): 'FilingOrganizationsTotalGrassrootsNontaxableAmt',
                        (1, 'AffiliatedGroupTotalAmt'): 'AffiliatedGroupTotalGrassrootsNontaxableAmt',
                    },
                    (7, 'TotLbbyngGrassrootMnsNonTxGrp'): {
                        # Key format: (Order, Element Tag)
                        (0, 'FilingOrganizationsTotalAmt'): 'FilingOrganizationsTotalTotLbbyngGrassrootMnsNonTxAmt',
                        (1, 'AffiliatedGroupTotalAmt'): 'AffiliatedGroupTotalTotLbbyngGrassrootMnsNonTxAmt',
                    },
                    (8, 'TotLbbyExpendMnsLbbyngNonTxGrp'): {
                        # Key format: (Order, Element Tag)
                        (0, 'FilingOrganizationsTotalAmt'): 'FilingOrganizationsTotalTotLbbyExpendMnsLbbyngNonTxAmt',
                        (1, 'AffiliatedGroupTotalAmt'): 'AffiliatedGroupTotalTotLbbyExpendMnsLbbyngNonTxAmt',
                    },
                    # Key format: (Order, Element Tag)
                    (9, 'Form4720FiledInd'): 'Form4720FiledInd'
                }
            },
            'Part II-B': {
                'Title': '',
                'Stipulation': 'Complete if the organization is exempt under section 501(c)(3) and has NOT filed Form 5768 (election under section 501(h)).',
                'OrgTypes': [r'501c3'],
                'Flat Content': {
                    (0, 'VolunteersInd'): 'VolunteersInd',
                    (1, 'PaidStaffOrManagementInd'): 'PaidStaffOrManagementInd',
                    (2, 'MediaAdvertisementsInd'): 'MediaAdvertisementsInd',
                    (3, 'MailingsMembersInd'): 'MailingsMembersInd',
                    (4, 'PublicationsOrBroadcastInd'): 'PublicationsOrBroadcastInd',
                    (5, 'GrantsOtherOrganizationsInd'): 'GrantsOtherOrganizationsInd',
                    (6, 'DirectContactLegislatorsInd'): 'DirectContactLegislatorsInd',
                    (7, 'RalliesDemonstrationsInd'): 'RalliesDemonstrationsInd',
                    (8, 'OtherActivitiesInd'): 'OtherActivitiesInd',
                    (9, 'NotDescribedSection501c3Ind'): 'NotDescribedSection501c3Ind',
                    (10, 'Form4720Filed4912TaxInd'): 'Form4720Filed4912TaxInd'
                }
            }
        }
    },
    'Part III': {
        'Title': '',
        'Stipulation': '',
        'OrgTypes': [],
        'Flat Content': {
            'Part III-A': {  
                'Title': '',
                'Stipulation': 'Complete if the organization is exempt under section 501(c)(4), section 501(c)(5), or section 501(c)(6).',
                'OrgTypes': [r'501c4'],
                'Flat Content': {
                    (0, 'SubstantiallyAllDuesNondedInd'): 'SubstantiallyAllDuesNondedInd',
                    (1, 'OnlyInHouseLobbyingInd'): 'OnlyInHouseLobbyingInd',
                    (2, 'AgreeCarryoverPriorYearInd'): 'AgreeCarryoverPriorYearInd'
                }          
            },
            'Part III-B': {
                'Title': '',
                'Stipulation': 'Complete if the organization is exempt under section 501(c)(4), section 501(c)(5), or section 501(c)(6) and if either (a) BOTH Part III-A, lines 1 and 2, are answered "No" OR (b) Part III-A, line 3, is answered \"Yes.\"',
                'OrgTypes': ['501c4'],
                'Flat Content': {}
            }
        }
    },
    'Part IV': {
        'Title': '',
        'Stipulation': 'Complete if the organization is exempt under section 501(c)(4), section 501(c)(5), or section 501(c)(6).',
        'OrgTypes': [r'501c4'],
        'Flat Content': {
            (0, 'SupplementalInformationDetail') : {
                (0, 'FormAndLineReferenceDesc') : 'FormAndLineReferenceDesc',
                (1, 'ExplanationTxt') : 'ExplanationTxt',
            }
        }
        
    }
}



In [47]:
# prep_for_download = {}



prep_for_download = {}
def recursive_formatting(_dict, _new_dict, c=0):
    # print(f'c={c}')
    for _key, _val in _dict.items():
        # print("\t"*c,_key, type(_key), type(_val))
        _str_key = str(_key)
        if type(_val) == dict:
            c += 1
            _new_dict[_str_key] = {}
            recursive_formatting(_val, _new_dict[_str_key], c)
        else:
            _new_dict[_str_key] = _val
            # print("\t"*(1 + c), "Not dict", repr(_val))

# recursive_formatting(schedule_c, prep_for_download)
# prep_for_download

# with open('IRS_Form_990_Formatting.json', mode='w') as f:    
#     f.write(json.dumps(prep_for_download, indent=4))

In [44]:
a = (1, 'abc')
str(a)

"(1, 'abc')"

In [48]:
def create_xml_target_mapping_dict(_target: str, _dict: dict, _accum: dict = {}):
    if _target in _dict.keys():
        _sub_dict = _dict[_target]
        if len(_sub_dict) > 0:
            create_xml_target_mapping_dict(_target, _sub_dict, _accum)
    else:
        _dict_keys = [_dict_key for _dict_key in _dict.keys()]
        if type(_dict_keys[0]) == tuple:
            # Sort by the integer within the key's tuple
            sorted_dict = dict(sorted(_dict.items(), key=lambda x: x[0][0]))
            for _dict_key, _dict_val in sorted_dict.items():
                if type(_dict_val) == dict:
                    create_xml_target_mapping_dict(_target, _dict_val, _accum)
                else:
                    _accum[_dict_key[1]] = _dict_val
        else:
            for _dict_val in _dict.values():
                create_xml_target_mapping_dict(_target, _dict_val, _accum)
    return _accum

schedule_c_flat_targets = []
# for part, subpart in schedule_c.items():
create_xml_target_mapping_dict('Flat Content', schedule_c)

{'PoliticalExpendituresAmt': 'PoliticalExpendituresAmt',
 'VolunteerHoursCnt': 'VolunteerHoursCnt',
 'Section4955OrganizationTaxAmt': 'Section4955OrganizationTaxAmt',
 'Section4955ManagersTaxAmt': 'Section4955ManagersTaxAmt',
 'Form4720FiledSection4955TaxInd': 'CorrectionMadeInd',
 'CorrectionMadeInd': 'CorrectionMadeInd',
 'Expended527ActivitiesAmt': 'Expended527ActivitiesAmt',
 'InternalFundsContributedAmt': 'InternalFundsContributedAmt',
 'TotalExemptFunctionExpendAmt': 'TotalExemptFunctionExpendAmt',
 'Form1120POLFiledInd': 'Form1120POLFiledInd',
 'FilingOrganizationsTotalAmt': 'FilingOrganizationsTotalTotLbbyExpendMnsLbbyngNonTxAmt',
 'AffiliatedGroupTotalAmt': 'AffiliatedGroupTotalTotLbbyExpendMnsLbbyngNonTxAmt',
 'Form4720FiledInd': 'Form4720FiledInd',
 'VolunteersInd': 'VolunteersInd',
 'PaidStaffOrManagementInd': 'PaidStaffOrManagementInd',
 'MediaAdvertisementsInd': 'MediaAdvertisementsInd',
 'MailingsMembersInd': 'MailingsMembersInd',
 'PublicationsOrBroadcastInd': 'Publicat

In [9]:
c4s = df.loc[df.OrgType == '501c4']
c4_ScB = c4s.loc[c4s.situation.str.contains('ScheduleB')]
for i, (situation, file) in enumerate(zip(c4_ScB.loc[:, 'situation'].values, c4_ScB.loc[:, 'file'].values)):
    print(i, situation, file)

0 A 501c4 filed a 990 containing IRS990ScheduleB in 2018. xml\2021\2021_01A\202110079349300101_public.xml
1 A 501c4 filed a 990EZ containing IRS990ScheduleB in 2018. xml\2019\2019_03A\201913199349208136_public.xml
2 A 501c4 filed a 990 containing IRS990ScheduleB in 2017. xml\2019\2019_01A\201902259349301420_public.xml
3 A 501c4 filed a 990EZ containing IRS990ScheduleB in 2017. xml\2019\2019_03A\201911359349202321_public.xml
4 A 501c4 filed a 990 containing IRS990ScheduleB in 2016. xml\2019\2019_02A\201910359349300131_public.xml
5 A 501c4 filed a 990EZ containing IRS990ScheduleB in 2016. xml\2019\2019_02A\201910209349200001_public.xml
6 A 501c4 filed a 990 containing IRS990ScheduleB in 2019. xml\2020\2020_04A\202021719349301472_public.xml
7 A 501c4 filed a 990EZ containing IRS990ScheduleB in 2019. xml\2020\2020_04A\202021709349201117_public.xml
8 A 501c4 filed a 990 containing IRS990ScheduleB in 2020. xml\2021\2021_01A\202143449349300144_public.xml
9 A 501c4 filed a 990EZ containing IRS

In [10]:
print(df.loc[df.IRS990ScheduleBSize > 1, 'ReturnTypeCd'].unique())
print(df.loc[df.IRS990ScheduleBSize > 1, 'OrgType'].unique())
df.loc[df.IRS990ScheduleBSize > 1].sort_values(by='IRS990ScheduleBSize', ascending=False).head()

<StringArray>
['990PF']
Length: 1, dtype: str
<StringArray>
['501c3']
Length: 1, dtype: str


,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
357,A 501c3 filed a 990PF containing IRS990Schedul...,472024662,THE SHAPIRO POGREBIN FOUNDATION,501c3,xml\2020\2020_03A\202013179349100631_public.xml,2019,990PF,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
476,A 501c3 filed a 990PF containing IRS990Schedul...,207097647,THE A PAUL AND CAROL C SCHAAP,501c3,xml\2021\2021_01A\202112579349100211_public.xml,2020,990PF,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
593,A 501c3 filed a 990PF containing IRS990Schedul...,306462100,THE HALCYON FOUNDATION,501c3,xml\2022\2022_01A\202222229349100502_public.xml,2021,990PF,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
821,A 501c3 filed a 990PF containing IRS990Schedul...,232686151,OBERKOTTER FOUNDATION,501c3,xml\2025\2025_05B\202531199349101103_public.xml,2023,990PF,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
940,A 501c3 filed a 990PF containing IRS990Schedul...,611913297,EVERY ORG,501c3,xml\2025\2025_10A\202502749349100700_public.xml,2024,990PF,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0


# IRS 990 Schedule B

In [11]:
# TODO: Find out what the GeneralRuleInd and Organization501c3ExemptPFInd flags mean
# TODO: Decide if the GeneralRuleInd and Organization501c3ExemptPFInd flags should be parsed
# TODO: Incorporate into larger parser to relate EIN

In [12]:
root = ET.parse(r'example_xmls/2020/2020_03A/202013179349100631_public.xml').getroot()
schedule_b = root.find(RETURN_DATA_PATH).find(f'{NAMESPACE}IRS990ScheduleB')
set([contributor_group.tag.replace(NAMESPACE, '') for contributor_group in schedule_b])

{'ContributorInformationGrp',
 'GeneralRuleInd',
 'NonCashPropertyContributionGrp',
 'Organization501c3ExemptPFInd'}

In [13]:
root.find('namespace-uri')

In [ ]:
import json
from ast import literal_eval
from Nonprofits.form990_parser_old import Form990Parser
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [67]:
f = Form990Parser('example_xmls/2020/2020_03A/202013179349100631_public.xml')
f._namespace
f.find_element('Filer')

<Element '{http://www.irs.gov/efile}Filer' at 0x000001E96FE55670>

In [ ]:
f = Form990Parser('example_xmls/2020/2020_03A/202013179349100631_public.xml')
f._namespace
f.find_element('Filer')
f.read_in_formatter('IRS_Form_990_Formatting.json')

{'Part I': {'Title': '',
  'Stipulation': 'Complete if the organization is exempt under section 501(c) or is a section 527 organization.',
  'OrgTypes': [],
  'Flat Content': {'Part I-A': {'Title': '',
    'Stipulation': '',
    'OrgTypes': ['501c\\d{1}', '527'],
    'Flat Content': {(0,
      'PoliticalExpendituresAmt'): 'PoliticalExpendituresAmt',
     (1, 'VolunteerHoursCnt'): 'VolunteerHoursCnt'}},
   'Part I-B': {'Title': '',
    'Stipulation': 'Complete if the organization is exempt under section 501(c)(3).',
    'OrgTypes': ['501c3'],
    'Flat Content': {(0,
      'Section4955OrganizationTaxAmt'): 'Section4955OrganizationTaxAmt',
     (1, 'Section4955ManagersTaxAmt'): 'Section4955ManagersTaxAmt',
     (2, 'Form4720FiledSection4955TaxInd'): 'CorrectionMadeInd',
     (3, 'CorrectionMadeInd'): 'CorrectionMadeInd'}},
   'Part I-C': {'Title': '',
    'Stipulation': 'Complete if the organization is exempt under section 501(c), except section 501(c)(3).',
    'OrgTypes': ['501c4'],
  

In [ ]:
d = json.loads()
new_dict = {}
def recursive_literal_formatter(_dict, _new_dict):
    for _key, _val in _dict.items():
        try:
            _key = literal_eval(_key)
        except (SyntaxError, ValueError) as error:
            # This throws when the literal value isn't quotted
            pass
        if type(_val) == dict:
            _new_dict[_key] = {}
            recursive_literal_formatter(_val, _new_dict[_key])
        else:
            _new_dict[_key] = _val
    return _new_dict

recursive_literal_formatter(d, new_dict)

{'Part I': {'Title': '',
  'Stipulation': 'Complete if the organization is exempt under section 501(c) or is a section 527 organization.',
  'OrgTypes': [],
  'Flat Content': {'Part I-A': {'Title': '',
    'Stipulation': '',
    'OrgTypes': ['501c\\d{1}', '527'],
    'Flat Content': {(0,
      'PoliticalExpendituresAmt'): 'PoliticalExpendituresAmt',
     (1, 'VolunteerHoursCnt'): 'VolunteerHoursCnt'}},
   'Part I-B': {'Title': '',
    'Stipulation': 'Complete if the organization is exempt under section 501(c)(3).',
    'OrgTypes': ['501c3'],
    'Flat Content': {(0,
      'Section4955OrganizationTaxAmt'): 'Section4955OrganizationTaxAmt',
     (1, 'Section4955ManagersTaxAmt'): 'Section4955ManagersTaxAmt',
     (2, 'Form4720FiledSection4955TaxInd'): 'CorrectionMadeInd',
     (3, 'CorrectionMadeInd'): 'CorrectionMadeInd'}},
   'Part I-C': {'Title': '',
    'Stipulation': 'Complete if the organization is exempt under section 501(c), except section 501(c)(3).',
    'OrgTypes': ['501c4'],
  

In [32]:
# a = list(d['Part I']['Flat Content']['Part I-A']['Flat Content'].keys())[0]
# print(a)
# literal_eval('PartI')

In [58]:
contributor_information_groups = schedule_b.findall(f'{NAMESPACE}ContributorInformationGrp')

contributor_data = []
for contributor_information_group in contributor_information_groups:
    contributor_content = {
        'EIN': 'TBD',
    }
    for info in contributor_information_group.iter():
        text = re.sub(r'\s', '', info.text)
        if len(text) == 0:
            continue
        field = re.sub(NAMESPACE, '', info.tag)
        contributor_content[field] = text
    contributor_data.append(contributor_content)

contributor_information_df = pd.DataFrame(contributor_data).head()
contributor_information_df.head()

,EIN,ContributorNum,BusinessNameLine1Txt,AddressLine1Txt,CityNm,StateAbbreviationCd,ZIPCd,TotalContributionsAmt,NoncashContributionInd,PersonContributionInd
0,TBD,1,DAVIDSHAPIRO&ABIGAILPOGREBIN,15EAST26THSTREET,NEWYORK,NY,10010,923,X,NaN
1,TBD,2,DAVIDSHAPIRO&ABIGAILPOGREBIN,15EAST26THSTREET,NEWYORK,NY,10010,579,X,NaN
2,TBD,3,DAVIDSHAPIRO&ABIGAILPOGREBIN,15EAST26THSTREET,NEWYORK,NY,10010,4080,X,NaN
3,TBD,4,DAVIDSHAPIRO&ABIGAILPOGREBIN,15EAST26THSTREET,NEWYORK,NY,10010,132,X,NaN
4,TBD,5,DAVIDSHAPIRO&ABIGAILPOGREBIN,15EAST26THSTREET,NEWYORK,NY,10010,756,X,NaN


In [57]:
non_cash_property_contribution_groups = schedule_b.findall(f'{NAMESPACE}NonCashPropertyContributionGrp')

property_contribution_data = []
for non_cash_property_contribution_group in non_cash_property_contribution_groups:
    property_contribution_content = {
        'EIN': 'TBD',
    }
    for info in non_cash_property_contribution_group.iter():
        text = re.sub(r'\s', '', info.text)
        if len(text) == 0:
            continue
        field = re.sub(NAMESPACE, '', info.tag)
        property_contribution_content[field] = text
    property_contribution_data.append(property_contribution_content)

non_cash_property_contribution_df = pd.DataFrame(property_contribution_data)
non_cash_property_contribution_df.head()

,EIN,ContributorNum,NoncashPropertyDesc,FairMarketValueAmt,ReceivedDt
0,TBD,1,478X8INC,923,2019-12-18
1,TBD,2,10AARON'SINC,579,2019-12-18
2,TBD,3,46ABBOTTLABORATORIES,4080,2019-12-18
3,TBD,4,8ABERCROMBIE&FITCHCO-CLA,132,2019-12-18
4,TBD,5,4ABIOMEDINC,756,2019-12-18


In [24]:
print(df.loc[df.IRS990ScheduleBSize == 1, 'ReturnTypeCd'].unique())
df.loc[df.IRS990ScheduleBSize == 1]

<StringArray>
['990', '990PF', '990EZ']
Length: 3, dtype: str


,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
0,A 501c3 filed a 990 containing IRS990 in 2018.,136227614,HADASSAH GROUP RETURN,501c3,xml\2019\2019_03A\201913199349306206_public.xml,2018,990,1.0,3652.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,A 501c3 filed a 990 containing IRS990ScheduleA...,135562401,YOUNG MEN'S CHRISTIAN ASSOCIATION RETIREMENT FUND,501c3,xml\2019\2019_03A\201913159349303926_public.xml,2018,990,1.0,267.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,A 501c3 filed a 990 containing IRS990ScheduleB...,452318579,CARIBBEAN STUDIES ASSOCIATIONINC,501c3,xml\2021\2021_01A\202101069349300000_public.xml,2018,990,1.0,190.0,1.0,...,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0
4,A 501c3 filed a 990 containing IRS990ScheduleF...,510198509,TIDES FOUNDATION,501c3,xml\2019\2019_03A\201913189349314251_public.xml,2018,990,1.0,274.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
5,A 501c3 filed a 990 containing IRS990ScheduleG...,810421425,ROCKY MOUNTAIN ELK FOUNDATION INC,501c3,xml\2019\2019_03A\201913179349302811_public.xml,2018,990,1.0,368.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1114,A 501c4 filed a 990EZ containing IRS990Schedul...,952873405,JOSLYN ADULT RECREATION CENTER OF CAMBRIA INC,501c4,xml\2026\2026_4A\202631109349200203_public.xml,2025,990EZ,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN
1115,A 501c4 filed a 990EZ containing IRS990Schedul...,161363462,WESTERN NEW YORK ASSOCIATION OF RETIRED LAW EN...,501c4,xml\2026\2026_3A\202640619349200719_public.xml,2025,990EZ,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN
1120,A 501c4 filed a 990EZ containing TransferPrsnl...,841417032,CARBONDALE AGRICULTURAL HERITAGE FUND,501c4,xml\2026\2026_4A\202631109349200428_public.xml,2025,990EZ,0.0,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN
1122,A 501c4 filed a 990EZ containing CompensationE...,813842947,TEJAS SILHOUETTE SHOOTERS,501c4,xml\2026\2026_3A\202600709349201710_public.xml,2025,990EZ,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0


# IRS 990 Schedule C

In [65]:
# TODO: Find out what the following flags mean:
    # Expended527ActivitiesAmt
    # Form1120POLFiledInd
    # InternalFundsContributedAmt
    # PoliticalExpendituresAmt
    # SupplementalInformationDetail
    # TotalExemptFunctionExpendAmt
# TODO: For the flags we want, parse them at the filer level
# TODO: Incorporate into larger parser to relate EIN

In [24]:
import json
from ast import literal_eval
import xml.etree.ElementTree as ET
from form990_parser_old import *
from form990_parser import Form990Parser
%load_ext autoreload
%autoreload 2
path = r'example_xmls/2019/2019_02A/201903159349302410_public.xml'
root = ET.parse(path).getroot()
SCHEDULE_C = root.find(RETURN_DATA_PATH).find(f'{NAMESPACE}IRS990ScheduleC')
set([SCHEDULE_C_subgroup.tag.replace(NAMESPACE, '') for SCHEDULE_C_subgroup in SCHEDULE_C])

{'Expended527ActivitiesAmt',
 'Form1120POLFiledInd',
 'InternalFundsContributedAmt',
 'PoliticalExpendituresAmt',
 'Section527PoliticalOrgGrp',
 'SupplementalInformationDetail',
 'TotalExemptFunctionExpendAmt'}

In [ ]:
f = Form990Parser('example_xmls/2020/2020_03A/202013179349100631_public.xml')
f._namespace
f.find_element('Filer')
formatter = f.read_in_formatter('IRS_Form_990_Formatting.json')

In [188]:
# from typeguard import check_type
# from typing import List

In [189]:
# def flat_content_formatting(_flat_content_key: tuple[int, str], _flat_content_val: str) -> tuple[str, str]:
#     return _flat_content_key[1], _flat_content_val

# def nested_content_formatting(_nested_content_dict: dict):
#     return _nested_content_dict['Nested Content']

# f.create_xml_target_mapping_list(nested_content, nested_content_formatting, formatter)

In [190]:
flat_content = dict[tuple[int, str], str]
nested_content = dict[str, list[dict[str, str | dict]]]

def flat_content_formatting(_flat_content_key: tuple[int, str], _flat_content_val: str) -> tuple[str, str]:
    return _flat_content_key[1], _flat_content_val

def nested_content_formatting(_nested_content_dict: dict):
    return _nested_content_dict['Nested Content']

c = f.parse_schedule_c(
    _ein='tbd',
    _schedule_c_root=SCHEDULE_C,
    _flat_targets=f.create_xml_target_mapping_dict(flat_content, flat_content_formatting, formatter),
    _nested_targets=f.create_xml_target_mapping_list(nested_content, nested_content_formatting, formatter)
)

In [191]:
c['Section527PoliticalOrgGrp']

,EIN,BusinessName,Address,City,StateCode,ZipCode,PaidInternalFunds,ContributionsReceivedAndDelivered
0,tbd,ABBY LEE,5370 ELMORE RD,FRUITLAND,ID,83619,250,NaN
1,tbd,AGENBROAD FOR SENATE,3615 PORTLAND AVE,NAMPA,ID,83686,250,NaN
2,tbd,ANGELA ROMERO,1098 S EMERY ST,SLC,UT,84104,250,NaN
3,tbd,BEDKE FOR LEGISLATURE,PO BOX 89,OAKLEY,ID,83346,250,NaN
4,tbd,BLANKSMA FOR IDAHO,PO BOX 843,MOUNTAIN HOME,ID,83647,250,NaN
...,...,...,...,...,...,...,...,...
64,tbd,SCOTT SYME,17498 ALLENDALE RD,WILDER,ID,83676,100,NaN
65,tbd,SENATOR KAREN MAYNE,5044 BANNOCK CIR,WEST VALLEY CITY,UT,84120,1500,NaN
66,tbd,UTAH HOUSE DEMOCRATIC LEADERSHIP COUNCIL,PO BOX 155,SLC,UT,84101,500,NaN
67,tbd,UTAH REPUBLICAN HOUSE ELECTION COMMITTEE,117 E SOUTH TEMPLE,SLC,UT,84111,2500,NaN


In [133]:
c['Section527PoliticalOrgGrp'].columns

Index(['EIN', 'BusinessNameLine1Txt', 'AddressLine1Txt', 'CityNm',
       'StateAbbreviationCd', 'ZIPCd', 'PaidInternalFundsAmt',
       'ContributionsRcvdDlvrAmt'],
      dtype='str')

In [193]:
c['Flat Content']

{'EIN': 'tbd',
 'PoliticalExpendituresAmt': '61369',
 'Expended527ActivitiesAmt': '269',
 'InternalFundsContributedAmt': '61100',
 'TotalExemptFunctionExpendAmt': '61369',
 'Form1120POLFiledInd': '1',
 'FormAndLineReferenceDesc': 'PART I-A, LINE 1:',
 'ExplanationTxt': 'SELECTHEALTH PARTICIPATES IN THE POLITICAL PROCESS BY PROVIDING SMALL AMOUNTS OF DIRECT CASH AND IN-KIND CONTRIBUTIONS TO STATE AND LOCAL CANDIDATES RUNNING FOR PUBLIC OFFICE.'}

In [64]:
section_527_political_org_groups = schedule_c.findall(f'{NAMESPACE}Section527PoliticalOrgGrp')
section_527_political_org_data = []
for section_527_political_org_group in section_527_political_org_groups:
    section_527_political_org_content = {
        'EIN': 'TBD',
    }
    for info in section_527_political_org_group.iter():
        text = re.sub(r'\s', '', info.text)
        if len(text) == 0:
            continue
        field = re.sub(NAMESPACE, '', info.tag)
        section_527_political_org_content[field] = text
    section_527_political_org_data.append(section_527_political_org_content)

section_527_political_org_df = pd.DataFrame(section_527_political_org_data)
section_527_political_org_df.head()

,EIN,BusinessNameLine1Txt,AddressLine1Txt,CityNm,StateAbbreviationCd,ZIPCd,PaidInternalFundsAmt,ContributionsRcvdDlvrAmt
0,TBD,ABBYLEE,5370ELMORERD,FRUITLAND,ID,83619,250,NaN
1,TBD,AGENBROADFORSENATE,3615PORTLANDAVE,NAMPA,ID,83686,250,NaN
2,TBD,ANGELAROMERO,1098SEMERYST,SLC,UT,84104,250,NaN
3,TBD,BEDKEFORLEGISLATURE,POBOX89,OAKLEY,ID,83346,250,NaN
4,TBD,BLANKSMAFORIDAHO,POBOX843,MOUNTAINHOME,ID,83647,250,NaN


# IRS 990 Schedule J

In [28]:
print(df.loc[(df.IRS990ScheduleJSize > 1) & (df.OrgType == '501c4'), 'ReturnTypeCd'].unique())
print(df.loc[(df.IRS990ScheduleJSize > 1) & (df.OrgType == '501c4'), 'OrgType'].unique())
df.loc[(df.IRS990ScheduleJSize > 1) & (df.OrgType == '501c4')].sort_values(by='IRS990ScheduleJSize', ascending=False).head()

<StringArray>
['990']
Length: 1, dtype: str
<StringArray>
['501c4']
Length: 1, dtype: str


,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
33,A 501c4 filed a 990 containing IRS990ScheduleJ...,941461312,DELTA DENTAL OF CALIFORNIA,501c4,xml\2019\2019_02A\201903179349301005_public.xml,2018,990,1.0,293.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
930,A 501c4 filed a 990 containing IRS990ScheduleJ...,912117699,BOYS & GIRLS CLUBS OF AMERICA,501c4,xml\2025\2025_10A\202532759349301648_public.xml,2024,990,1.0,627.0,0.0,...,0.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0
466,A 501c4 filed a 990 containing IRS990ScheduleJ...,941461312,DELTA DENTAL OF CALIFORNIA,501c4,xml\2021\2021_01A\202113159349301046_public.xml,2020,990,1.0,292.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0
575,A 501c4 filed a 990 containing IRS990 in 2021.,912117699,Boys & Girls Clubs of America,501c4,xml\2022\2022_01A\202202849349300240_public.xml,2021,990,1.0,644.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
581,A 501c4 filed a 990 containing IRS990ScheduleI...,912117699,Boys & Girls Clubs of America,501c4,xml\2022\2022_01A\202202849349300240_public.xml,2021,990,1.0,644.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0


# Schedule I

In [11]:
schedule = 'IRS990ScheduleI'
cols = ["situation", "EIN", "Name", "OrgType", "file", "TaxYr", "ReturnTypeCd", schedule, f"{schedule}Size"]
schedule_I_df = c4s.loc[c4s.loc[:, schedule] == 1, cols].sort_values(by=f"{schedule}Size", ascending=False)
schedule_I_df

,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990ScheduleI,IRS990ScheduleISize
925,A 501c4 filed a 990 containing IRS990ScheduleI...,951985500,AARP,501c4,xml\2025\2025_11A\202503159349305695_public.xml,2024,990,1.0,692.0
581,A 501c4 filed a 990 containing IRS990ScheduleI...,912117699,Boys & Girls Clubs of America,501c4,xml\2022\2022_01A\202202849349300240_public.xml,2021,990,1.0,527.0
586,A 501c4 filed a 990 containing IRS990ScheduleJ...,912117699,Boys & Girls Clubs of America,501c4,xml\2022\2022_01A\202202849349300240_public.xml,2021,990,1.0,527.0
575,A 501c4 filed a 990 containing IRS990 in 2021.,912117699,Boys & Girls Clubs of America,501c4,xml\2022\2022_01A\202202849349300240_public.xml,2021,990,1.0,527.0
461,A 501c4 filed a 990 containing IRS990ScheduleI...,951985500,AARP,501c4,xml\2021\2021_01A\202113149349304491_public.xml,2020,990,1.0,523.0
...,...,...,...,...,...,...,...,...,...
578,A 501c4 filed a 990 containing IRS990ScheduleD...,237401870,SMITHTOWN VOLUNTEER EXEMPT FIREMAN'S,501c4,xml\2022\2022_01A\202221269349300247_public.xml,2021,990,1.0,3.0
251,A 501c4 filed a 990 containing IRS990ScheduleI...,381753186,MISS MICHIGAN SCHOLARSHIP PAGEANT,501c4,xml\2019\2019_02A\201910749349301431_public.xml,2016,990,1.0,2.0
705,A 501c4 filed a 990 containing IRS990ScheduleN...,471854208,Liberty Lodge 58 Real Estate Corp,501c4,xml\2024\2024_02A\202420439349302312_public.xml,2022,990,1.0,2.0
1047,A 501c4 filed a 990 containing IRS990ScheduleJ...,521750248,FAMILIES AGAINST MANDATORY MINIMUMS,501c4,xml\2026\2026_4A\202640909349300219_public.xml,2025,990,1.0,2.0


In [16]:
schec_i_elems = set()
for file in schedule_I_df.loc[:, 'file'].values:
    print(file)
    file = re.sub(r'^xml', r'example_xmls', file)
    file = re.sub(r'\\', '/', file)
    schec_i = ET.parse(file).getroot().find(f'.//{NAMESPACE}IRS990ScheduleI')
    
    curr_content = set([elem.tag.replace(NAMESPACE, '') for elem in schec_i.iter()])
    
    diff = curr_content - schec_i_elems
    schec_i_elems |= curr_content
    for new_elem in diff:
        print(new_elem)
    print("="*20)

xml\2025\2025_11A\202503159349305695_public.xml
Total501c3OrgCnt
StateAbbreviationCd
GrantRecordsMaintainedInd
TotalOtherOrgCnt
IRCSectionDesc
AddressLine2Txt
ZIPCd
ExplanationTxt
PurposeOfGrantTxt
RecipientBusinessName
RecipientTable
BusinessNameLine1Txt
CashGrantAmt
FormAndLineReferenceDesc
NonCashAssistanceAmt
RecipientEIN
CityNm
SupplementalInformationDetail
AddressLine1Txt
IRS990ScheduleI
USAddress
xml\2022\2022_01A\202202849349300240_public.xml
RecipientCnt
GrantTypeTxt
GrantsOtherAsstToIndivInUSGrp
xml\2022\2022_01A\202202849349300240_public.xml
xml\2022\2022_01A\202202849349300240_public.xml
xml\2021\2021_01A\202113149349304491_public.xml
xml\2021\2021_01A\202113079349301516_public.xml
xml\2020\2020_02A\202003219349319750_public.xml
xml\2020\2020_04A\202022819349301582_public.xml
xml\2020\2020_04A\202022819349301582_public.xml
xml\2024\2024_02A\202420439349302002_public.xml
xml\2025\2025_01A\202510159349303021_public.xml
xml\2025\2025_10A\202532759349301648_public.xml
xml\2019\

In [37]:
schedule_stipulation = "Complete if the organization answered 'Yes,' on Form 990, Part IV, line 21 or 22."
schedule_i_formatter = {
    "Part I": {
        "Title": "General Information on Grants and Assistance",
        "Stipulation": "",
        "OrgTypes": [], # none mentioned
        "Flat Content": {
            (0, 'GrantRecordsMaintainedInd'): "GrantRecordsMaintainedInd"
        }
    },
    "Part II": {
        "Title": "Grants and Other Assistance to Domestic Organizations and Domestic Governments.",
        "Stipulation": "Complete if the organization answered 'Yes' on Form 990, Part IV, line 21, for any recipient that received more than $5,000.",
        "OrgTypes": [],
        "Flat Content": {
            (0, 'Total501c3OrgCnt'): "Total501c3OrgCnt",
            (0, 'TotalOtherOrgCnt'): "TotalOtherOrgCnt"
        },
        "Nested Content": [
            {
                "Group": "RecipientTable",
                "Column Name Mapper": {
                    "BusinessNameLine1Txt": "BusinessName",
                    "BusinessNameLine2Txt": "BusinessNameLine2",
                    "AddressLine1Txt": "Address",
                    "AddressLine2Txt": "AddressLine2",
                    "CityNm": "City",
                    "StateAbbreviationCd": "StateCode",
                    "ZIPCd": "ZipCode",
                    "RecipientEIN": "RecipientEIN",
                    "IRCSectionDesc": "IRCSectionDesc",
                    "CashGrantAmt": "CashGrant",
                    "NonCashAssistanceAmt": "NonCashAssistance",
                    "NonCashAssistanceDesc": "NonCashAssistanceDesc",
                    "PurposeOfGrantTxt": "PurposeOfGrant"
                }   
            }
        ]
    },
    "Part III": {
        "Title": "Grants and Other Assistance to Domestic Individuals.",
        "Stipulation": "Complete if the organization answered 'Yes' on Form 990, Part IV, line 22.",
        "OrgTypes": [],
        "Flat Content": {},
        "Nested Content": [
            {
                "Group": "GrantsOtherAsstToIndivInUSGrp",
                "Column Name Mapper": {
                    "GrantTypeTxt": "GrantType",
                    "RecipientCnt": "Recipients",
                    "CashGrantAmt": "CashGrant",
                    "ValuationMethodUsedDesc": "ValuationMethodUsedDesc",
                    "NonCashAssistanceAmt": "NonCashAssistance",
                    "NonCashAssistanceDesc": "NonCashAssistanceDesc"
                }
            }
        ]
    },
    "Part IV": {
        "Title": "Supplemental Information.",
        "Stipulation": "Complete if the organization is exempt under section 501(c)(4), section 501(c)(5), or section 501(c)(6).",
        "Description": "Provide the information required in Part I, line 2; Part III, column (b); and any other additional information.",
        "OrgTypes": [
            "501c4"
        ],
        "Flat Content": {
            (0, 'Explanation'): "MonitoringProceduresAndOtherExplanations"
        }
    }
}

In [38]:
full_formatter = {
    'Schedule C': {
        'Title': 'Political Campaign and Lobbying Activities',
        "Stipulation": """If the organization answered "Yes" on Form 990, Part IV, Line 3, or Form 990-EZ, Part V, line 46 (Political Campaign Activities), then
\t• Section 501(c)(3) organizations: Complete Parts I-A and B. Do not complete Part I-C.
\t• Section 501(c) (other than section 501(c)(3)) organizations: Complete Parts I-A and C below. Do not complete Part I-В.
\t• Section 527 organizations: Complete Part I-A only.
If the organization answered "Yes" on Form 990, Part IV, Line 4, or Form 990-EZ, Part VI, line 47 (Lobbying Activities), then
\t• Section 501(c)(3) organizations that have filed Form 5768 (election under section 501(h)): Complete Part II-A. Do not complete Part II-B.
\t• Section 501(c)(3) organizations that have NOT filed Form 5768 (election under section 501(h)): Complete Part II-B. Do not complete Part II-A.
If the organization answered "Yes" on Form 990, Part IV, Line 5 (Proxy Tax) (see separate instructions) or Form 990-EZ, Part V, line 35c
(Proxy Tax) (see separate instructions), then
\t• Section 501(c)(4), (5), or (6) organizations: Complete Part III.
""",
        "OrgTypes": [
            "501c3", "501c4", "527"
        ],
        "Content": formatter
    },
    'Schedule I': {
        'Title': 'Grants and Other Assistance to Organizations, Governments and Individuals in the United States',
        "Stipulation": "Complete if the organization answered 'Yes,' on Form 990, Part IV, line 21 or 22.",
        "OrgTypes": [
            "501c3", "501c4", "527"
        ],
        "Content": schedule_i_formatter
    }
}

In [49]:
prep_for_download = {}

recursive_formatting(full_formatter, prep_for_download)
prep_for_download

with open('IRS_Form_990_Formatting.json', mode='w') as f:    
    f.write(json.dumps(prep_for_download, indent=4))

In [52]:
f.read_in_formatter('IRS_Form_990_Formatting.json')

{'Schedule C': {'Title': 'Political Campaign and Lobbying Activities',
  'Stipulation': 'If the organization answered "Yes" on Form 990, Part IV, Line 3, or Form 990-EZ, Part V, line 46 (Political Campaign Activities), then\n\t• Section 501(c)(3) organizations: Complete Parts I-A and B. Do not complete Part I-C.\n\t• Section 501(c) (other than section 501(c)(3)) organizations: Complete Parts I-A and C below. Do not complete Part I-В.\n\t• Section 527 organizations: Complete Part I-A only.\nIf the organization answered "Yes" on Form 990, Part IV, Line 4, or Form 990-EZ, Part VI, line 47 (Lobbying Activities), then\n\t• Section 501(c)(3) organizations that have filed Form 5768 (election under section 501(h)): Complete Part II-A. Do not complete Part II-B.\n\t• Section 501(c)(3) organizations that have NOT filed Form 5768 (election under section 501(h)): Complete Part II-B. Do not complete Part II-A.\nIf the organization answered "Yes" on Form 990, Part IV, Line 5 (Proxy Tax) (see separat

In [ ]:
schec_i_elems = set()
for file in schedule_I_df.loc[:, 'file'].values:
    print(file)
    file = re.sub(r'^xml', r'example_xmls', file)
    file = re.sub(r'\\', '/', file)
    schec_i = ET.parse(file).getroot().find(f'.//{NAMESPACE}IRS990ScheduleI')
    
    curr_content = set([elem.tag.replace(NAMESPACE, '') for elem in schec_i.iter()])
    
    diff = curr_content - schec_i_elems
    schec_i_elems |= curr_content
    for new_elem in diff:
        print(new_elem)
    print("="*20)

# json formatter

In [135]:
group_elems = [
        'Form990PartVIISectionAGrp',
        'ContractorCompensationGrp',
        'Section527PoliticalOrgGrp',
        'AvgLobbyingNontaxableAmountGrp',
        'AvgTotalLobbyingExpendGrp',
        'AvgGrassrootsNontaxableGrp',
        'AvgGrassrootsLobbyingExpendGrp',
        'SupplementalInformationDetail',
        'RecipientTable',
        'GrantsOtherAsstToIndivInUSGrp',        
    ]

common_mappings = {
    'BusinessNameLine1Txt': 'BusinessName',
    'BusinessNameLine2Txt': 'BusinessNameLine2',
    'AddressLine1Txt': 'Address',
    'BusinessNameLine2Txt': 'AddressLine2',
    'CityNm': 'City',
    'StateAbbreviationCd': 'StateCode',
    'ZIPCd': 'ZipCode',
    'PhoneNum': 'Phone',
    # Form990PartVIISectionAGrp
    'TitleTxt': 'Title',
    "IndividualTrusteeOrDirectorInd": "IndividualTrusteeOrDirector",
    "InstitutionalTrusteeInd": "InstitutionalTrustee",
    "OfficerInd": "Officer",
    "KeyEmployeeInd": "KeyEmployee",
    "HighestCompensatedEmployeeInd": "HighestCompensatedEmployee",
    "FormerInd": "FormerEmployee",
    "ReportableCompFromOrgAmt": "ReportableCompFromOrg",
    "ReportableCompFromRltdOrgAmt": "ReportableCompFromRltdOrg",
    "OtherCompensationAmt": "OtherCompensation",
    # Section527PoliticalOrgGrp
    "PaidInternalFundsAmt": "PaidInternalFunds",
    "ContributionsRcvdDlvrAmt": "ContributionsReceivedAndDelivered",
    # Others
    "Minus3Amt": "Minus3",
    "Minus2Amt": "Minus2",
    "Minus1Amt": "Minus1",
    "Amt": "",
    "TotalAmt": "Total",
    # RecipientTable
    "NonCashAssistanceAmt": "NonCashAssistance",
    "PurposeOfGrantTxt": "PurposeOfGrant",
    # GrantsOtherAsstToIndivInUSGrp
    "GrantTypeTxt": "GrantType",
    "RecipientCnt": "Recipients",
    "CashGrantAmt": "CashGrant",
    "ValuationMethodUsedDesc": "ValuationMethodUsedDesc",
    "NonCashAssistanceAmt": "NonCashAssistance",
    "NonCashAssistanceDesc": "NonCashAssistanceDesc"
}

parsed_sections = [
    'ReturnHeader'
]

parsed_elements = [
    "PrincipalOfficerNm",
    "GrossReceiptsAmt",
    "GroupReturnForAffiliatesInd",
    "AllAffiliatesIncludedInd",
    "WebsiteAddressTxt",
    "FormationYr",
    "LegalDomicileStateCde",
    "ActivityOrMissionDesc",
    "VotingMembersGoverningBodyCnt",
    "VotingMembersIndependentCnt",
    "TotalEmployeeCnt",
    "TotalGrossUBIAmtI",
    "CYContributionsGrantsAmt",
    "CYTotalRevenueAmtt",
    "CYGrantsAndSimilarPaidAmt",
    "CYBenefitsPaidToMembersAmt",
    "CYSalariesCompEmpBnftPaidAmt",
    "CYTotalProfFndrsngExpnsAmt",
    "CYTotalFundraisingExpenseAmt",
    "CYOtherExpensesAmt",
    "CYTotalExpensesAmt",
    "CYRevenuesLessExpensesAmtt",
    "TotalAssetsBOYAmt",
    "TotalAssetsEOYAmt",
    "TotalLiabilitiesBOYAmt",
    "TotalLiabilitiesEOYAmt",
    "NetAssetsOrFundBalancesBOYAmt",
    "NetAssetsOrFundBalancesEOYAmtr"
]

In [268]:
def is_group_header(_element: ET.Element) -> bool:
    # Identify parent element with no real content
    if _element.text is None:
        return True
    if len(re.sub(r'\s', '', _element.text)) == 0:
        return True
    return False

def get_elem_dtype(elem_tag, elem_text):
    if elem_text in ['X', 'false', 'true'] or elem_tag[-3:] == 'Ind':
        return 'bool'
    elif elem_tag[-3:] in ['Amt', 'Cnt']:
        return 'float'
    else:
        return 'str'

def get_mapping(tag, common_mappings=common_mappings):
    try:
        return common_mappings[tag]
    except KeyError:
        if tag[-3] in ['Amt', 'Cnt', 'Txt', 'Ind']:
            return tag[:-3]
        return tag

def format_elem(elem, parent='Return', group_elems=group_elems, common_mappings=common_mappings):
    tag = re.sub('{http://www.irs.gov/efile}', '', elem.tag)
    if parent != 'Return':
        parent = re.sub('{http://www.irs.gov/efile}', '', parent.tag)
    
    mapping = get_mapping(tag, common_mappings=common_mappings)

    
    if is_group_header(elem):
        if tag in group_elems:
            element_type = 'Group'
        else:
            element_type = None
            pass
        return {
                tag: {
                    'Mapping': mapping,
                    'DataType': None,
                    'IsHeader': True,
                    'ElementType': element_type,
                    'IsParsed': False,
                    'Parent': parent,
                    'SectionPartsAffected': None,
                    'Children': {}
                }
            }
    
    if tag in group_elems:
        element_type = 'Group'
    else:
        element_type = 'Singular'
    
    return {
            tag: {
                'Mapping': mapping,
                'DataType': get_elem_dtype(tag, elem.text),
                'IsHeader': False,
                'ElementType': element_type,
                'IsParsed': False,
                'Parent': parent,
                'SectionPartsAffected': None,
                'Children': None
            }
        }



def recursive(elem, parent_elem = 'Return', d = None, prev_d = {}, parsed_sections=parsed_sections, parsed_elements=parsed_elements):
    if d is None:
        d = {}
    for sub_elem in elem:
        elem_dict = format_elem(sub_elem, elem)
        e_tag = list(elem_dict.keys())[0]

        d[e_tag] = elem_dict[e_tag]
        try:
            d[e_tag]['IsParsed'] = prev_d[elem_dict[e_tag]['Parent']]['IsParsed'] or e_tag in parsed_sections or e_tag in parsed_elements
        except KeyError:
            d[e_tag]['IsParsed'] = e_tag in parsed_sections or e_tag in parsed_elements
        if elem_dict[e_tag]['IsHeader']:
            # print(s, elem, ':')
            # print(s, '-', sub_elem)
            recursive(sub_elem, elem, d = d[e_tag]['Children'], prev_d=d)
            continue
        recursive(sub_elem, elem, d)
    return d

In [269]:
file = r'xml\2019\2019_02A\201903199349310630_public.xml'
# file = r'xml\2025\2025_11B\202543179349300049_public.xml'

In [270]:
D = recursive(ET.parse(Path(file)).getroot())
print(json.dumps(D, indent=4))

{
    "ReturnHeader": {
        "Mapping": "ReturnHeader",
        "DataType": null,
        "IsHeader": true,
        "ElementType": null,
        "IsParsed": true,
        "Parent": "Return",
        "SectionPartsAffected": null,
        "Children": {
            "ReturnTs": {
                "Mapping": "ReturnTs",
                "DataType": "str",
                "IsHeader": false,
                "ElementType": "Singular",
                "IsParsed": true,
                "Parent": "ReturnHeader",
                "SectionPartsAffected": null,
                "Children": null
            },
            "TaxPeriodEndDt": {
                "Mapping": "TaxPeriodEndDt",
                "DataType": "str",
                "IsHeader": false,
                "ElementType": "Singular",
                "IsParsed": true,
                "Parent": "ReturnHeader",
                "SectionPartsAffected": null,
                "Children": null
            },
            "PreparerFirmGrp": {
     

In [271]:
a = {
    "DescribedInSection501c3Ind": {
            "Mapping": "DescribedInSection501c3",
            "ScheduleParts": ["ScheduleA"]
          },
          "ScheduleBRequiredInd": {
            "Mapping": "ScheduleBRequired",
            "ScheduleParts": ["ScheduleB"]
          },
          "PoliticalCampaignActyInd": {
            "Mapping": "PoliticalCampaignActy",
            "ScheduleParts": ["IRS990ScheduleC_Part I"]
          },
          "LobbyingActivitiesInd": {
            "Mapping": "LobbyingActivities",
            "ScheduleParts": ["IRS990ScheduleC_Part II"]
          },
          "SubjectToProxyTaxInd": {
            "Mapping": "SubjectToProxyTax",
            "ScheduleParts": ["IRS990ScheduleC_Part III"]
          },
          "DonorAdvisedFundInd": {
            "Mapping": "DonorAdvisedFund",
            "ScheduleParts": ["IRS990ScheduleD_Part I"]
          },
          "MoreThan5000KToOrgInd": {
            "Mapping": "MoreThan5KToOrg",
            "ScheduleParts": [
              "IRS990ScheduleF_Part II",
              "IRS990ScheduleF_Part IV"
            ]
          },
          "MoreThan5000KToIndividualsInd": {
            "Mapping": "MoreThan5KToIndividuals",
            "ScheduleParts": [
              "IRS990ScheduleF_Part III",
              "IRS990ScheduleF_Part IV"
            ]
          },
          "ProfessionalFundraisingInd": {
            "Mapping": "ProfessionalFundraising",
            "ScheduleParts": ["IRS990ScheduleG_Part I"]
          },
          "FundraisingActivitiesInd": {
            "Mapping": "FundraisingActivities",
            "ScheduleParts": ["IRS990ScheduleG_Part II"]
          },
          "GrantsToOrganizationsInd": {
            "Mapping": "GrantsToOrganizations",
            "ScheduleParts": [
              "IRS990ScheduleI_Part I",
              "IRS990ScheduleI_Part II"
            ]
          },
          "GrantsToIndividualsInd": {
            "Mapping": "GrantsToIndividuals",
            "ScheduleParts": [
              "IRS990ScheduleI_Part I",
              "IRS990ScheduleI_Part III"
            ]
          },
          "ScheduleJRequiredInd": {
            "Mapping": "ScheduleJRequired",
            "ScheduleParts": ["ScheduleJ"]
          },
          "GrantToRelatedPersonInd": {
            "Mapping": "GrantToRelatedPersonInd",
            "ScheduleParts": ["IRS990ScheduleL_Part III"]
          },
          "BusinessRlnWithOrgMemInd": {
            "Mapping": "BusinessRlnWithOrgMemInd",
            "ScheduleParts": ["IRS990ScheduleL_Part IV"]
          },
          "BusinessRlnWithFamMemInd": {
            "Mapping": "BusinessRlnWithFamMemInd",
            "ScheduleParts": ["IRS990ScheduleL_Part IV"]
          },
          "BusinessRlnWithOfficerEntInd": {
            "Mapping": "BusinessRlnWithOfficerEntInd",
            "ScheduleParts": []
          },
          "TerminateOperationsInd": {
            "Mapping": "TerminateOperationsInd",
            "ScheduleParts": ["IRS990ScheduleN_Part I"]
          },
          "PartialLiquidationInd": {
            "Mapping": "PartialLiquidationInd",
            "ScheduleParts": ["IRS990ScheduleN_Part II"]
          },
          "DisregardedEntityInd": {
            "Mapping": "DisregardedEntityInd",
            "ScheduleParts": ["IRS990ScheduleR_Part I"]
          },
          "RelatedEntityInd": {
            "Mapping": "RelatedEntityInd",
            "ScheduleParts": [
              "IRS990ScheduleR_Part II",
              "IRS990ScheduleR_Part III",
              "IRS990ScheduleR_Part IV",
              "IRS990ScheduleR_Part V"
            ]
          },
          "RelatedOrganizationCtrlEntInd": {
            "Mapping": "RelatedOrganizationCtrlEntInd",
            "ScheduleParts": []
          },
          "TransactionWithControlEntInd": {
            "Mapping": "TransactionWithControlEntInd",
            "ScheduleParts": ["IRS990ScheduleR_Part V"]
          },
          "TrnsfrExmptNonChrtblRltdOrgInd": {
            "Mapping": "TrnsfrExmptNonChrtblRltdOrgInd",
            "ScheduleParts": ["IRS990ScheduleR_Part V"]
          },
          "ActivitiesConductedPrtshpInd": {
            "Mapping": "ActivitiesConductedPrtshpInd",
            "ScheduleParts": ["IRS990ScheduleR_Part VI"]
          },
          "ScheduleORequiredInd": {
            "Mapping": "ScheduleORequiredInd",
            "ScheduleParts": ["ScheduleO"]
          }
        }


for k, v in a.items():
    # a[k] = v['ScheduleParts']
    # print(v)
    print(k)
    D["ReturnData"]["Children"]["IRS990"]["Children"][k]['SectionPartsAffected'] = v['ScheduleParts']
    D["ReturnData"]["Children"]["IRS990"]["Children"][k]['IsParsed'] = True

DescribedInSection501c3Ind
ScheduleBRequiredInd
PoliticalCampaignActyInd
LobbyingActivitiesInd
SubjectToProxyTaxInd
DonorAdvisedFundInd
MoreThan5000KToOrgInd
MoreThan5000KToIndividualsInd
ProfessionalFundraisingInd
FundraisingActivitiesInd
GrantsToOrganizationsInd
GrantsToIndividualsInd
ScheduleJRequiredInd
GrantToRelatedPersonInd
BusinessRlnWithOrgMemInd
BusinessRlnWithFamMemInd
BusinessRlnWithOfficerEntInd
TerminateOperationsInd
PartialLiquidationInd
DisregardedEntityInd
RelatedEntityInd
RelatedOrganizationCtrlEntInd
TransactionWithControlEntInd
TrnsfrExmptNonChrtblRltdOrgInd
ActivitiesConductedPrtshpInd
ScheduleORequiredInd


In [272]:
with open('test.json', encoding='utf-8', mode='w') as f:
    f.write(json.dumps(D, indent=4))

In [112]:
file_lookup = pd.read_csv('example_usecase_lookup.csv')
file_lookup.sample(5)

,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
269,A 501c3 filed a 990PF containing OtherLiabilit...,743044117,URBAN ADVANCEMENT CORPORATION,501c3,xml\2019\2019_03A\201913189349103216_public.xml,2016,990PF,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
913,A 501c3 filed a 990 containing IRS990ScheduleK...,550643304,West Virginia University Hospitals Inc,501c3,xml\2025\2025_11B\202543189349303119_public.xml,2024,990,1.0,304.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0
1011,A 501c4 filed a 990EZ containing TransferPrsnl...,992438757,BRUNDAGE MOUNTAIN FIRE PROTECTION,501c4,xml\2025\2025_12A\202503259349200800_public.xml,2024,990EZ,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
521,A 501c3 filed a 990PF containing EmployeeCompe...,226029397,The Robert Wood Johnson Foundation,501c3,xml\2021\2021_01A\202143159349102389_public.xml,2020,990PF,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
198,A 501c3 filed a 990PF containing EmployeeCompe...,131628151,CARNEGIE CORPORATION OF NEW YORK,501c3,xml\2019\2019_03A\201911909349100511_public.xml,2017,990PF,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [ ]:
df = file_lookup.loc[(file_lookup.ReturnTypeCd == '990')]
elements = []
for file in df.file.unique():
    xml = ET.parse(Path(file))

xml\2019\2019_03A\201913199349306206_public.xml
xml\2019\2019_03A\201913159349303926_public.xml
xml\2021\2021_01A\202101069349300000_public.xml
xml\2019\2019_01A\201901349349305675_public.xml
xml\2019\2019_03A\201913189349314251_public.xml
xml\2019\2019_03A\201913179349302811_public.xml
xml\2020\2020_02A\202010489349300101_public.xml
xml\2019\2019_03A\201912769349301126_public.xml
xml\2020\2020_03A\202011979349302981_public.xml
xml\2020\2020_02A\202010459349301146_public.xml
xml\2020\2020_04A\202021909349301007_public.xml
xml\2019\2019_02A\201903199349310630_public.xml
xml\2019\2019_03A\201912849349301956_public.xml
xml\2021\2021_01A\202110149349300326_public.xml
xml\2019\2019_03A\201913049349300001_public.xml
xml\2019\2019_02A\201903179349303965_public.xml
xml\2019\2019_02A\201903189349304325_public.xml
xml\2020\2020_04A\202021969349302832_public.xml
xml\2019\2019_03A\201912949349301406_public.xml
xml\2020\2020_03A\202011979349304051_public.xml
xml\2019\2019_02A\201903059349301130_pub

In [198]:
file_lookup.loc[(file_lookup.ReturnTypeCd == '990')].sort_values(by='IRS990Size', ascending=False)

,situation,EIN,Name,OrgType,file,TaxYr,ReturnTypeCd,IRS990,IRS990Size,IRS990ScheduleA,...,ExplanOfLegisPoliticalActvts,ExplanOfLegisPoliticalActvtsSize,AffiliateListing,AffiliateListingSize,AffiliatedGroupAttachment,AffiliatedGroupAttachmentSize,BorrowedFundsElection,BorrowedFundsElectionSize,TaxUnderSection511Statement,TaxUnderSection511StatementSize
0,A 501c3 filed a 990 containing IRS990 in 2018.,136227614,HADASSAH GROUP RETURN,501c3,xml\2019\2019_03A\201913199349306206_public.xml,2018,990,1.0,3652.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
919,A 501c4 filed a 990 containing IRS990 in 2024.,237028846,NATIONAL ASSOCIATION FOR THE ADVANCEMENT,501c4,xml\2025\2025_11B\202543179349300049_public.xml,2024,990,1.0,3591.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0.0
433,A 501c3 filed a 990 containing IRS990 in 2020.,136227614,HADASSAH GROUP RETURN,501c3,xml\2021\2021_01A\202113149349307586_public.xml,2020,990,1.0,3057.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
781,A 501c3 filed a 990 containing IRS990 in 2023.,208295721,UPMC GROUP,501c3,xml\2025\2025_05A\202511339349304441_public.xml,2023,990,1.0,1296.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0
11,A 501c3 filed a 990 containing IRS990ScheduleJ...,208295721,UPMC GROUP,501c3,xml\2020\2020_04A\202021909349301007_public.xml,2018,990,1.0,1056.0,1.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32,A 501c4 filed a 990 containing IRS990ScheduleL...,831559193,Fairness for Fantasy Sports Louisiana,501c4,xml\2019\2019_02A\201903109349302960_public.xml,2018,990,1.0,157.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
349,A 501c4 filed a 990 containing ReasonableCause...,621374275,WINDERMERE VOLUNTEER FIRE,501c4,xml\2020\2020_04A\202022819349301412_public.xml,2019,990,1.0,155.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
254,A 501c4 filed a 990 containing IRS990ScheduleL...,460793813,PUBLIC INTEGRITY ALLIANCE INC,501c4,xml\2019\2019_02A\201910369349301216_public.xml,2016,990,1.0,154.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
257,A 501c4 filed a 990 containing ReasonableCause...,460793813,PUBLIC INTEGRITY ALLIANCE INC,501c4,xml\2019\2019_02A\201910369349301216_public.xml,2016,990,1.0,154.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [185]:
with open('test.json', mode='r') as f:
    gen_formatter = json.loads(f.read())['ReturnData']['Children']['IRS990']['Children']
gen_formatter

{'AddressChangeInd': {'Mapping': 'AddressChange',
  'Datatype': 'bool',
  'IsHeader': False,
  'ElementType': 'Singular',
  'IsParsed': False,
  'Parent': 'IRS990',
  'SectionPartsAffected': None,
  'Children': None},
 'PrincipalOfficerNm': {'Mapping': 'PrincipalOfficerNm',
  'Datatype': 'str',
  'IsHeader': False,
  'ElementType': 'Singular',
  'IsParsed': True,
  'Parent': 'IRS990',
  'SectionPartsAffected': None,
  'Children': None},
 'USAddress': {'Mapping': 'USAddress',
  'DataType': None,
  'IsHeader': True,
  'ElementType': None,
  'IsParsed': False,
  'Parent': 'IRS990',
  'SectionPartsAffected': None,
  'Children': {'Address': {'Mapping': 'Address',
    'Datatype': 'str',
    'IsHeader': False,
    'ElementType': 'Singular',
    'IsParsed': False,
    'Parent': 'USAddress',
    'SectionPartsAffected': None,
    'Children': None},
   'City': {'Mapping': 'City',
    'Datatype': 'str',
    'IsHeader': False,
    'ElementType': 'Singular',
    'IsParsed': False,
    'Parent': 'USA

In [184]:
gen_formatter

{'Mapping': 'IRS990',
 'DataType': None,
 'IsHeader': True,
 'ElementType': None,
 'IsParsed': False,
 'Parent': 'ReturnData',
 'SectionPartsAffected': None,
 'Children': {'AddressChangeInd': {'Mapping': 'AddressChange',
   'Datatype': 'bool',
   'IsHeader': False,
   'ElementType': 'Singular',
   'IsParsed': False,
   'Parent': 'IRS990',
   'SectionPartsAffected': None,
   'Children': None},
  'PrincipalOfficerNm': {'Mapping': 'PrincipalOfficerNm',
   'Datatype': 'str',
   'IsHeader': False,
   'ElementType': 'Singular',
   'IsParsed': True,
   'Parent': 'IRS990',
   'SectionPartsAffected': None,
   'Children': None},
  'USAddress': {'Mapping': 'USAddress',
   'DataType': None,
   'IsHeader': True,
   'ElementType': None,
   'IsParsed': False,
   'Parent': 'IRS990',
   'SectionPartsAffected': None,
   'Children': {'Address': {'Mapping': 'Address',
     'Datatype': 'str',
     'IsHeader': False,
     'ElementType': 'Singular',
     'IsParsed': False,
     'Parent': 'USAddress',
     'S

In [262]:
file = r'xml\2019\2019_02A\201903199349310630_public.xml'
position = 1
f1 = []
f1_set = set()
f1_mapper = {}
for elem in ET.parse(Path(file)).getroot().find('.//{http://www.irs.gov/efile}IRS990'):
    tag = re.sub('{http://www.irs.gov/efile}', '', elem.tag)
    if tag in f1_set and tag in group_elems:
        continue
    f1_set |= set([tag])
    f1.append((position, tag))
    f1_mapper[tag] = position
    position += 1

file = r'xml\2025\2025_11B\202543179349300049_public.xml'
position = 1
f2 = []
f2_set = set()
f2_mapper = {}
for elem in ET.parse(Path(file)).getroot().find('.//{http://www.irs.gov/efile}IRS990'):
    tag = re.sub('{http://www.irs.gov/efile}', '', elem.tag)
    if tag in f2_set and tag in group_elems:
        continue
    f2_set |= set([tag])
    f2.append((position, tag))
    f2_mapper[tag] = position
    position += 1
    
    
file = r'xml\2019\2019_03A\201913159349303926_public.xml'
position = 1
f3 = []
f3_set = set()
f3_mapper = {}
for elem in ET.parse(Path(file)).getroot().find('.//{http://www.irs.gov/efile}IRS990'):
    tag = re.sub('{http://www.irs.gov/efile}', '', elem.tag)
    if tag in f3_set and tag in group_elems:
        continue
    f3_set |= set([tag])
    f3.append((position, tag))
    f3_mapper[tag] = position
    position += 1

In [267]:
f1

[(1, 'AddressChangeInd'),
 (2, 'PrincipalOfficerNm'),
 (3, 'USAddress'),
 (4, 'GrossReceiptsAmt'),
 (5, 'GroupReturnForAffiliatesInd'),
 (6, 'AllAffiliatesIncludedInd'),
 (7, 'Organization501c3Ind'),
 (8, 'WebsiteAddressTxt'),
 (9, 'TypeOfOrganizationCorpInd'),
 (10, 'FormationYr'),
 (11, 'LegalDomicileStateCd'),
 (12, 'ActivityOrMissionDesc'),
 (13, 'VotingMembersGoverningBodyCnt'),
 (14, 'VotingMembersIndependentCnt'),
 (15, 'TotalEmployeeCnt'),
 (16, 'TotalVolunteersCnt'),
 (17, 'TotalGrossUBIAmt'),
 (18, 'PYContributionsGrantsAmt'),
 (19, 'CYContributionsGrantsAmt'),
 (20, 'PYProgramServiceRevenueAmt'),
 (21, 'CYProgramServiceRevenueAmt'),
 (22, 'PYInvestmentIncomeAmt'),
 (23, 'CYInvestmentIncomeAmt'),
 (24, 'PYOtherRevenueAmt'),
 (25, 'CYOtherRevenueAmt'),
 (26, 'PYTotalRevenueAmt'),
 (27, 'CYTotalRevenueAmt'),
 (28, 'PYGrantsAndSimilarPaidAmt'),
 (29, 'CYGrantsAndSimilarPaidAmt'),
 (30, 'PYBenefitsPaidToMembersAmt'),
 (31, 'CYBenefitsPaidToMembersAmt'),
 (32, 'PYSalariesCompEmpBn

In [2]:
A = [
    'GrantsToDomesticOrgsGrp',
    'GrantsToDomesticIndividualsGrp',
    'BenefitsToMembersGrp',
    'CompCurrentOfcrDirectorsGrp',
    'CompDisqualPersonsGrp',
    'OtherSalariesAndWagesGrp',
    'PensionPlanContributionsGrp',
    'OtherEmployeeBenefitsGrp',
    'PayrollTaxesGrp',
    'FeesForServicesLegalGrp',
    'FeesForServicesAccountingGrp',
    'FeesForSrvcInvstMgmntFeesGrp',
    'FeesForServicesOtherGrp',
    'AdvertisingGrp',
    'OfficeExpensesGrp',
    'InformationTechnologyGrp',
    'OccupancyGrp',
    'TravelGrp',
    'ConferencesMeetingsGrp',
    'DepreciationDepletionGrp',
    'InsuranceGrp',
    'OtherExpensesGrp',
    'AllOtherExpensesGrp',
    'TotalFunctionalExpensesGrp',
]

d = {}
for a in A:
    print(a)
    d[a] = {
        "Mapping": a,
        "DataType": None,
        "IsHeader": True,
        "ElementType": None,
        "IsParsed": False,
        "Parent": "IRS990",
        "SectionPartsAffected": None,
        "Children": {
            "TotalAmt": {
                "Mapping": "Total",
                "DataType": "float",
                "IsHeader": False,
                "ElementType": "Singular",
                "IsParsed": False,
                "Parent": a,
                "SectionPartsAffected": None,
                "Children": None
            },
            "ProgramServicesAmt": {
                "Mapping": "ProgramServices",
                "DataType": "float",
                "IsHeader": False,
                "ElementType": "Singular",
                "IsParsed": False,
                "Parent": a,
                "SectionPartsAffected": None,
                "Children": None
            },
            "ManagementAndGeneralAmt": {
                "Mapping": "ManagementAndGeneral",
                "DataType": "float",
                "IsHeader": False,
                "ElementType": "Singular",
                "IsParsed": False,
                "Parent": a,
                "SectionPartsAffected": None,
                "Children": None
            },
            "FundraisingAmt": {
                "Mapping": "Fundraising",
                "DataType": "float",
                "IsHeader": False,
                "ElementType": "Singular",
                "IsParsed": False,
                "Parent": a,
                "SectionPartsAffected": None,
                "Children": None
            }
        }
    }

GrantsToDomesticOrgsGrp
GrantsToDomesticIndividualsGrp
BenefitsToMembersGrp
CompCurrentOfcrDirectorsGrp
CompDisqualPersonsGrp
OtherSalariesAndWagesGrp
PensionPlanContributionsGrp
OtherEmployeeBenefitsGrp
PayrollTaxesGrp
FeesForServicesLegalGrp
FeesForServicesAccountingGrp
FeesForSrvcInvstMgmntFeesGrp
FeesForServicesOtherGrp
AdvertisingGrp
OfficeExpensesGrp
InformationTechnologyGrp
OccupancyGrp
TravelGrp
ConferencesMeetingsGrp
DepreciationDepletionGrp
InsuranceGrp
OtherExpensesGrp
AllOtherExpensesGrp
TotalFunctionalExpensesGrp


In [6]:
import json
print(json.dumps(d, indent=4))

{
    "GrantsToDomesticOrgsGrp": {
        "Mapping": "GrantsToDomesticOrgsGrp",
        "DataType": null,
        "IsHeader": true,
        "ElementType": null,
        "IsParsed": false,
        "Parent": "IRS990",
        "SectionPartsAffected": null,
        "Children": {
            "TotalAmt": {
                "Mapping": "Total",
                "DataType": "float",
                "IsHeader": false,
                "ElementType": "Singular",
                "IsParsed": false,
                "Parent": "GrantsToDomesticOrgsGrp",
                "SectionPartsAffected": null,
                "Children": null
            },
            "ProgramServicesAmt": {
                "Mapping": "ProgramServices",
                "DataType": "float",
                "IsHeader": false,
                "ElementType": "Singular",
                "IsParsed": false,
                "Parent": "GrantsToDomesticOrgsGrp",
                "SectionPartsAffected": null,
                "Children": null


In [ ]:
a = {
    "Return": {
        "990": {
            "Part 0": {
                "PrincipalOfficerNm": "PrincipalOfficer",
                # "USAddress": {
                    # "AddressLine1Txt": "PrincipalOfficerAddress",
                    # "AddressLine2Txt": "PrincipalOfficerAddressLine2",
                    # "CityNm": "PrincipalOfficerCity",
                    # "StateAbbreviationCd": "PrincipalOfficerStateCode",
                    # "ZIPCd": "PrincipalOfficerZipCode"
                # },
                "GrossReceiptsAmt": "GrossReceipts",
                "GroupReturnForAffiliatesInd": "GroupReturnForAffiliates",
                # TODO: Group exemption number
                "AllAffiliatesIncludedInd": "AllAffiliatesIncluded",
                # "Organization501cInd": "Organization501cInd",
                "WebsiteAddressTxt": "Website",
                # "TypeOfOrganizationCorpInd": "TypeOfOrganizationCorpInd",
                "FormationYr": "FormationYear",
                "LegalDomicileStateCd": "LegalDomicileStateCode"
            },
            "Part I": {
                "Title": "Summary",
                "Part I-A": {
                    "Title": "Activities & Govemance",
                    "Flat Content": {
                        "ActivityOrMissionDesc": "ActivityOrMission",
                        "VotingMembersGoverningBodyCnt": "GoverningBodyVotingMembers",
                        "VotingMembersIndependentCnt": "IndependentVotingMembers",
                        "TotalEmployeeCnt": "TotalEmployees",
                        "TotalGrossUBIAmt": "TotalGrossUBI"
                    }
                },
                "Part I-B": {
                    "Title": "Revenue",
                    "Flat Content": {
                        "CYContributionsGrantsAmt": "ContributionsGrants",
                        # "PYContributionsGrantsAmt": "PreviousYearContributionsGrants",
                        "CYTotalRevenueAmt": "TotalRevenue",
                        # "PYTotalRevenueAmt": "PreviousYearTotalRevenue"
                    }
                },
                "Part I-C": {
                    "Title": "Expenses",
                    "Flat Content": {
                        # "PreviousYearGrantsAndSimilarPaidAmt": "PYGrantsAndSimilarPaid",
                        "CYGrantsAndSimilarPaidAmt": "GrantsAndSimilarPaid",
                        # "PreviousYearBenefitsPaidToMembersAmt": "PYBenefitsPaidToMembers",
                        "CYBenefitsPaidToMembersAmt": "BenefitsPaidToMembers",
                        # "PreviousYearSalariesCompEmpBnftPaidAmt": "PYSalariesCompEmpBnftPaid",
                        "CYSalariesCompEmpBnftPaidAmt": "SalariesCompEmpBnftPaid",
                        # "PreviousYearTotalProfFndrsngExpnsAmt": "PYTotalProfFndrsngExpns",
                        "CYTotalProfFndrsngExpnsAmt": "TotalProfFndrsngExpns",
                        "CYTotalFundraisingExpenseAmt": "TotalFundraisingExpense",
                        # "PreviousYearOtherExpensesAmt": "PYOtherExpenses",
                        "CYOtherExpensesAmt": "OtherExpenses",
                        # "PreviousYearTotalExpensesAmt": "PYTotalExpenses",
                        "CYTotalExpensesAmt": "TotalExpenses",
                        # "PreviousYearRevenuesLessExpensesAmt": "PYRevenuesLessExpenses",
                        "CYRevenuesLessExpensesAmt": "RevenuesLessExpenses"
                    }
                },
                "Part I-D": {
                    "Title": "Net Assets or Fund Balances",
                    "Flat Content": {
                        "TotalAssetsBOYAmt": "TotalAssetsBeginningOfYear",
                        "TotalAssetsEOYAmt": "TotalAssetsEndOfYear",
                        "TotalLiabilitiesBOYAmt": "TotalLiabilitiesBeginningOfYear",
                        "TotalLiabilitiesEOYAmt": "TotalLiabilitiesEndOfYear",
                        "NetAssetsOrFundBalancesBOYAmt": "NetAssetsOrFundBalancesBeginningOfYear",
                        "NetAssetsOrFundBalancesEOYAmt": "NetAssetsOrFundBalancesEndOfYear"
                    }
                }
            },
            "Part II": {
                "Title": "Signature Block"
            },
            "Part III": {
                "Title": "Statement of Program Service Accomplishments"
            },
            "Part IV": {
                "Title": "Checklist of Required Schedules",
                "Flat Content": {
                    "DescribedInSection501c3Ind": {
                        "Mapping": "DescribedInSection501c3",
                        "SectionParts": ["ScheduleA"]
                    },
                    "ScheduleBRequiredInd": {
                        "Mapping": "ScheduleBRequired",
                        "SectionParts": ["ScheduleB"]
                    }, 
                    "PoliticalCampaignActyInd": {
                        "Mapping": "PoliticalCampaignActy",
                        "SectionParts": ["IRS990ScheduleC_Part I"]
                    },
                    "LobbyingActivitiesInd": {
                        "Mapping": "LobbyingActivities",
                        "SectionParts": ["IRS990ScheduleC_Part II"]
                    },
                    "SubjectToProxyTaxInd": {
                        "Mapping": "SubjectToProxyTax",
                        "SectionParts": ["IRS990ScheduleC_Part III"]
                    },
                    "DonorAdvisedFundInd": {
                        "Mapping": "DonorAdvisedFund",
                        "SectionParts": ["IRS990ScheduleD_Part I"]
                    },
                    
                    # <ConservationEasementsInd>false</ConservationEasementsInd>
                    # <CollectionsOfArtInd>false</CollectionsOfArtInd>
                    # <CreditCounselingInd>false</CreditCounselingInd>
                    # <TempOrPermanentEndowmentsInd>false</TempOrPermanentEndowmentsInd>
                    # <ReportLandBuildingEquipmentInd referenceDocumentId="DOCIDSCHEDD">true</ReportLandBuildingEquipmentInd>
                    # <ReportInvestmentsOtherSecInd>false</ReportInvestmentsOtherSecInd>
                    # <ReportProgramRelatedInvstInd>false</ReportProgramRelatedInvstInd>
                    # <ReportOtherAssetsInd>false</ReportOtherAssetsInd>
                    # <ReportOtherLiabilitiesInd>false</ReportOtherLiabilitiesInd>
                    # <IncludeFIN48FootnoteInd>false</IncludeFIN48FootnoteInd>
                    # <IndependentAuditFinclStmtInd referenceDocumentId="DOCIDSCHEDD">true</IndependentAuditFinclStmtInd>
                    # <ConsolidatedAuditFinclStmtInd>false</ConsolidatedAuditFinclStmtInd>
                    # <SchoolOperatingInd>false</SchoolOperatingInd>
                    # "ForeignOfficeInd": "ForeignOfficeInd",
                    # "ForeignActivitiesInd": "ForeignActivitiesInd",
                    
                    "MoreThan5000KToOrgInd": {
                        "Mapping": "MoreThan5KToOrg",
                        "SectionParts": ["IRS990ScheduleF_Part II","IRS990ScheduleF_Part IV"]
                    },
                    "MoreThan5000KToIndividualsInd": {
                        "Mapping": "MoreThan5KToIndividuals",
                        "SectionParts": ["IRS990ScheduleF_Part III","IRS990ScheduleF_Part IV"]
                    },
                    "ProfessionalFundraisingInd": {
                        "Mapping": "ProfessionalFundraising",
                        "SectionParts": ["IRS990ScheduleG_Part I"]
                    },
                    "FundraisingActivitiesInd": {
                        "Mapping": "FundraisingActivities",
                        "SectionParts": ["IRS990ScheduleG_Part II"]
                    },
                    # OperateHospitalInd ["IRS990ScheduleG_Part III"]
                    "GrantsToOrganizationsInd": {
                        "Mapping": "GrantsToOrganizations",
                        "SectionParts": ["IRS990ScheduleI_Part I", "IRS990ScheduleI_Part II"]
                    },
                    "GrantsToIndividualsInd": {
                        "Mapping": "GrantsToIndividuals",
                        "SectionParts": ["IRS990ScheduleI_Part I", "IRS990ScheduleI_Part III"]
                    },
                    "ScheduleJRequiredInd": {
                        "Mapping": "ScheduleJRequired",
                        "SectionParts": ["ScheduleJ"]
                    }, 
                    # <TaxExemptBondsInd>false</TaxExemptBondsInd>
                    # <InvestTaxExemptBondsInd>false</InvestTaxExemptBondsInd>
                    # <EscrowAccountInd>false</EscrowAccountInd>
                    # <OnBehalfOfIssuerInd>false</OnBehalfOfIssuerInd>
                    # <EngagedInExcessBenefitTransInd>false</EngagedInExcessBenefitTransInd>
                    # <PYExcessBenefitTransInd>false</PYExcessBenefitTransInd>
                    # <LoanOutstandingInd referenceDocumentId="DOCID204329740">true</LoanOutstandingInd>
                    
                    
                    "GrantToRelatedPersonInd": {
                        "Mapping": "GrantToRelatedPersonInd",
                        "ScheduleParts": ["IRS990ScheduleL_Part III"]
                    },
                    # Q28 - Org party to business transaction w/:
                    # Q28a - current former officer, director... --> If yes, Schedule L, Part IV
                    "BusinessRlnWithOrgMemInd": {
                        "Mapping": "BusinessRlnWithOrgMemInd",
                        "ScheduleParts": ["IRS990ScheduleL_Part IV"]
                    },
                    # Q28b - family member or anything in 28a --> If yes, Schedule L, Part IV
                    "BusinessRlnWithFamMemInd": {
                        "Mapping": "BusinessRlnWithFamMemInd",
                        "ScheduleParts": ["IRS990ScheduleL_Part IV"]
                    },
                    # Q28c - 35% controlled entity or 28a or 28b --> If yes, Schedule L, Part IV
                    "BusinessRlnWithOfficerEntInd": {
                        "Mapping": "BusinessRlnWithOfficerEntInd",
                        "ScheduleParts": []
                    },
                    # Schedule M
                    # <DeductibleNonCashContriInd referenceDocumentId="DOCIDSCHEDM">true</DeductibleNonCashContriInd>
                    # <DeductibleArtContributionInd>false</DeductibleArtContributionInd>
                    # Q31 - Did org terminate --> If yes, Schedule N, Part I
                    "TerminateOperationsInd": {
                        "Mapping": "TerminateOperationsInd",
                        "ScheduleParts": ["IRS990ScheduleN_Part I"]
                    },
                    # Q32 - >25% assets --> If yes, Schedule N, Part II
                    "PartialLiquidationInd": {
                        "Mapping": "PartialLiquidationInd",
                        "ScheduleParts": ["IRS990ScheduleN_Part II"]
                    },
                    # Q33 - Own 100% of diresgarded entity --> If yes, Schedule R, Part I
                    "DisregardedEntityInd": {
                        "Mapping": "DisregardedEntityInd",
                        "ScheduleParts": ["IRS990ScheduleR_Part I"]
                    },
                    # Q34 - Related to other tax-exempt or taxable entity
                    # --> If yes, Schedule R, Part II, III, or IV, and Part V line 1
                    "RelatedEntityInd": {
                        "Mapping": "RelatedEntityInd",
                        "ScheduleParts": ["IRS990ScheduleR_Part II", "IRS990ScheduleR_Part III", "IRS990ScheduleR_Part IV", "IRS990ScheduleR_Part V"]
                    },
                    # 35a - Control of another entity --> If yes, answer 35b
                    "RelatedOrganizationCtrlEntInd": {
                        "Mapping": "RelatedOrganizationCtrlEntInd",
                        "ScheduleParts": []
                    },
                    # 35b - Engage in transaction w/Controlled entity --> If yes, Schedule R, Part V, Line 2
                    "TransactionWithControlEntInd": {
                        "Mapping": "TransactionWithControlEntInd",
                        "ScheduleParts": ["IRS990ScheduleR_Part V"]
                    },
                    # Q36 - (Only for 501c3s) Transfers to exempt non-charitable related org
                    # --> If yes, Schedule R, Part V, line 2
                    "TrnsfrExmptNonChrtblRltdOrgInd": {
                        "Mapping": "TrnsfrExmptNonChrtblRltdOrgInd",
                        "ScheduleParts": ["IRS990ScheduleR_Part V"]
                    },
                    # Q37 - Org conducted >5% of activities through unrelated entity --> If yes, Schedule R, Part VI
                    "ActivitiesConductedPrtshpInd": {
                        "Mapping": "ActivitiesConductedPrtshpInd",
                        "ScheduleParts": ["IRS990ScheduleR_Part VI"]
                    },
                    # Q38 - Did org complete Schedule O; All orgs should
                    "ScheduleORequiredInd": {
                        "Mapping": "ScheduleORequiredInd",
                        "ScheduleParts": ["ScheduleO"]
                    },
                },
            },
            "Part V": {
                "Title": "Statements Regarding Other IRS Filings and Tax Compliance",
                # Already have employee count in summary section
                # "Flat Content": {
                #     "EmployeeCnt": "Employees"
                # }
            },
            "Part VI": {
                "Title": "Governance, Management, and Disclosure",
                "Flat Content": {
                    "Part VI-A": {
                        "Title": " Governing Body and Management",
                        "Flat Content": {
                            # We can get this in Part I-A
                            # "GoverningBodyVotingMembersCnt": "GoverningBodyVotingMembers",
                            # "IndependentVotingMemberCnt": "IndependentVotingMembers",
                            "DelegationOfMgmtDutiesInd": "DelegationOfManagementDuties",
                            "ChangeToOrgDocumentsInd": "ChangeToOrgDocuments",
                            "MaterialDiversionOrMisuseInd": "MaterialDiversionOrMisuse",
                            "MembersOrStockholdersInd": "MembersOrStockholders",
                            "ElectionOfBoardMembersInd": "ElectionOfBoardMembers",
                            "DecisionsSubjectToApprovaInd": "DecisionsSubjectToApproval",
                            "MinutesOfGoverningBodyInd": "GoverningBodyMeetingMinutes",
                            "MinutesOfCommitteesInd": "CommitteesMeetingMinutes",
                            "OfficerMailingAddressInd": "OfficerNotFoundAtMailingAddress",
                        }
                    },
                    "Part VI-B": {
                        "Title": "Policies",
                    },
                    "Part VI-C": {
                        "Title": "Disclosure",
                        "Flat Content": {
                            # TODO: Find examples for 'Own Website", "Another's Website", and 'Other (explain in Schedule O)
                            "UponRequestInd": "990AvailableAt"
                        }
                    }
                }
            },
            "Part VII": {
                "Part VII-A": {
                    "Title": "Officers, Directors, Trustees, Key Employees, and Highest Compensated Employees",
                    "Nested Content": [
                        {
                            "Group": "Form990PartVIISectionAGrp",
                            "Column Name Mapper": {
                                "PersonNm": "Name",
                                "TitleTxt": "Title",
                                "AverageHoursPerWeekRt": "AverageHoursPerWeekRt",
                                # ! Missing the average hours per week for related
                                "IndividualTrusteeOrDirectorInd": "IndividualTrusteeOrDirector",
                                "InstitutionalTrusteeInd": "InstitutionalTrustee",
                                "OfficerInd": "Officer",
                                # ! Guessing names for next three indicators
                                "KeyEmployeeInd": "KeyEmployee",
                                "HighestCompensatedEmployeeInd": "HighestCompensatedEmployee",
                                "FormerInd": "FormerEmployee",
                                "ReportableCompFromOrgAmt": "ReportableCompFromOrg",
                                "ReportableCompFromRltdOrgAmt": "ReportableCompFromRltdOrg",
                                "OtherCompensationAmt": "OtherCompensation"
                            }
                        }
                    ]
                },
                "Part VII-B": {
                    "Title": "Independent Contractors",
                    "Nested Content": [
                        {
                            "Group": "CntrctRcvdGreaterThan100KCnt",
                            "Column Name Mapper": {
                                # "PersonNm": "Name",
                                # "TitleTxt": "Title",
                                # "AverageHoursPerWeekRt": "Address",
                                # # ! Missing the average hours per week for related
                                # "IndividualTrusteeOrDirectorInd": "IndividualTrusteeOrDirector",
                                # "InstitutionalTrusteeInd": "InstitutionalTrustee
                                # "OfficerInd": "Officer",
                                # # ! Guessing names for next three indicators
                                # "KeyEmployeeInd": "KeyEmployee",
                                # "HighestCompensatedEmployeeInd": "HighestCompensatedEmployee"
                                # "FormerInd": "FormerEmployee",
                                # "StateAbbreviationCd": "StateCode",
                                # "ReportableCompFromOrgAmt": "ZipCode",
                                # "ReportableCompFromRltdOrgAmt": "RecipientEIN",
                                # "OtherCompensationAmt": "IRCSectionDesc",
                            }
                        }
                    ]
                },
            },
            "Part VIII": {
                "Title": "Statement of Revenue"
            },
            "Part IX": {
                "Title": "Statement of Functional Expenses",
                "OrgTypes": ["501c3", "501c4"]
            },
            "Part X": {
                "Title": "Balance Sheet"
            },
            "Part XI": {
                "Title": "Reconciliation of Net Assets"
            },
            "Part XII": {
                "Title": "Financial Statements and Reporting"
            },
        }
    }
}